# Big Data Pipeline: U.S. Border Crossing Analysis

## 1. Environment Setup
This section initializes Apache Spark, MongoDB, and required Python libraries.

In [7]:
# Initialize Spark
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BorderCrossingAnalysis") \
    .getOrCreate()

print("Spark is working!")

Spark is working!


In [8]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["border_crossing_db"]

print("MongoDB connected!")

MongoDB connected!


In [9]:
import pandas as pd
import matplotlib.pyplot as plt

print("Libraries loaded!")

Libraries loaded!


## 2. Data Loading and Initial Inspection
In this section, the dataset is loaded and inspected to understand its structure, size, and key attributes.

In [10]:
df = spark.read.csv("data/Border_Crossing_Entry_Data.csv", header=True, inferSchema=True)

print("Data loaded successfully!")

Data loaded successfully!


In [11]:
df.show(5)

+-----------+---------+---------+----------------+--------+-------+-----+--------+---------+--------------------+
|  Port Name|    State|Port Code|          Border|    Date|Measure|Value|Latitude|Longitude|               Point|
+-----------+---------+---------+----------------+--------+-------+-----+--------+---------+--------------------+
|    Hidalgo|    Texas|     2305|US-Mexico Border|Jan 2026|  Buses|  640|  26.095|  -98.271|POINT (-98.271092...|
|Brownsville|    Texas|     2301|US-Mexico Border|Jan 2026|  Buses|  264|  25.952|  -97.401|POINT (-97.40067 ...|
|    Warroad|Minnesota|     3423|US-Canada Border|Dec 2025|  Buses|    9|  48.999|  -95.377|POINT (-95.376555...|
|      Alcan|   Alaska|     3104|US-Canada Border|Nov 2025| Trucks|  547|  62.615| -141.001|POINT (-141.00144...|
|     Laredo|    Texas|     2304|US-Mexico Border|Jul 2025|  Buses| 2546|    27.5|  -99.507|POINT (-99.507412...|
+-----------+---------+---------+----------------+--------+-------+-----+--------+------

In [12]:
df.printSchema()

root
 |-- Port Name: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Port Code: integer (nullable = true)
 |-- Border: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Measure: string (nullable = true)
 |-- Value: integer (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Point: string (nullable = true)



In [13]:
print(df.count())

273391


In [14]:
print(len(df.columns))

10


In [15]:
for col in df.columns:
    print(col)

Port Name
State
Port Code
Border
Date
Measure
Value
Latitude
Longitude
Point


In [16]:
print("Total columns:", len(df.columns))

Total columns: 10


In [17]:
print("Column names:")
for col in df.columns:
    print(col)

Column names:
Port Name
State
Port Code
Border
Date
Measure
Value
Latitude
Longitude
Point


In [18]:
df.describe().show()

+-------+---------+----------+------------------+----------------+--------+--------------+------------------+------------------+------------------+--------------------+
|summary|Port Name|     State|         Port Code|          Border|    Date|       Measure|             Value|          Latitude|         Longitude|               Point|
+-------+---------+----------+------------------+----------------+--------+--------------+------------------+------------------+------------------+--------------------+
|  count|   273391|    273387|            273391|          273391|  273391|        273391|            273391|            273387|            273387|              273387|
|   mean|     NULL|      NULL|  2447.97364946176|            NULL|    NULL|          NULL| 41979.96001697203|43.909012114695585|-99.81721089518044|                NULL|
| stddev|     NULL|      NULL|1199.7471110625095|            NULL|    NULL|          NULL|180977.99503700226| 8.183531831499531| 18.23658657198981|        

In [19]:
from pyspark.sql.functions import col, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df.columns
]).show()

+---------+-----+---------+------+----+-------+-----+--------+---------+-----+
|Port Name|State|Port Code|Border|Date|Measure|Value|Latitude|Longitude|Point|
+---------+-----+---------+------+----+-------+-----+--------+---------+-----+
|        0|    4|        0|     0|   0|      0|    0|       4|        4|    4|
+---------+-----+---------+------+----+-------+-----+--------+---------+-----+



## 3. Data Cleaning and Feature Engineering
In this section, the dataset is cleaned and new features such as year and month are extracted from the date column.

In [36]:
import pyspark.sql.functions as F

df_clean = spark.read.csv("data/Border_Crossing_Entry_Data.csv", header=True, inferSchema=True)

df_clean = df_clean.select(
    "Port Name",
    "State",
    "Border",
    "Date",
    "Measure",
    "Value"
).dropna()

df_clean.select("Date").show(5, False)

+--------+
|Date    |
+--------+
|Jan 2026|
|Jan 2026|
|Dec 2025|
|Nov 2025|
|Jul 2025|
+--------+
only showing top 5 rows


In [37]:
df_clean = df_clean.withColumn(
    "Date",
    F.to_date(
        F.concat(F.lit("01 "), F.col("Date")),
        "dd MMM yyyy"
    )
)

df_clean.select("Date").show(5, False)

+----------+
|Date      |
+----------+
|2026-01-01|
|2026-01-01|
|2025-12-01|
|2025-11-01|
|2025-07-01|
+----------+
only showing top 5 rows


In [38]:
df_clean = df_clean.withColumn("Year", F.year(F.col("Date")))
df_clean = df_clean.withColumn("Month", F.month(F.col("Date")))

df_clean.select("Date", "Year", "Month").show(5)

+----------+----+-----+
|      Date|Year|Month|
+----------+----+-----+
|2026-01-01|2026|    1|
|2026-01-01|2026|    1|
|2025-12-01|2025|   12|
|2025-11-01|2025|   11|
|2025-07-01|2025|    7|
+----------+----+-----+
only showing top 5 rows


In [40]:
import pyspark.sql.functions as F

df_clean = spark.read.csv("data/Border_Crossing_Entry_Data.csv", header=True, inferSchema=True)

df_clean = df_clean.dropna()

df_clean = df_clean.select(
    "Port Name",
    "State",
    "Border",
    "Date",
    "Measure",
    "Value"
)

df_clean = df_clean.withColumn(
    "Date",
    F.to_date(
        F.concat(F.lit("01 "), F.trim(F.col("Date"))),
        "dd MMM yyyy"
    )
)

df_clean = df_clean.withColumn("Year", F.year(F.col("Date")))
df_clean = df_clean.withColumn("Month", F.month(F.col("Date")))

df_clean.show(20)

+-----------+------------+----------------+----------+--------------+-----+----+-----+
|  Port Name|       State|          Border|      Date|       Measure|Value|Year|Month|
+-----------+------------+----------------+----------+--------------+-----+----+-----+
|    Hidalgo|       Texas|US-Mexico Border|2026-01-01|         Buses|  640|2026|    1|
|Brownsville|       Texas|US-Mexico Border|2026-01-01|         Buses|  264|2026|    1|
|    Warroad|   Minnesota|US-Canada Border|2025-12-01|         Buses|    9|2025|   12|
|      Alcan|      Alaska|US-Canada Border|2025-11-01|        Trucks|  547|2025|   11|
|     Laredo|       Texas|US-Mexico Border|2025-07-01|         Buses| 2546|2025|    7|
|       Roma|       Texas|US-Mexico Border|2025-07-01|         Buses|   67|2025|    7|
|     Calais|       Maine|US-Canada Border|2025-04-01|Bus Passengers|  318|2025|    4|
|    Warroad|   Minnesota|US-Canada Border|2025-04-01|        Trucks|  433|2025|    4|
|       Roma|       Texas|US-Mexico Border|